# 6. Opportunity and Efficiency Metrics

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `05_scoring_engine.ipynb` first.

Raw fantasy points (`05_scoring_engine.ipynb`) tell you what happened. `nuclearff.metrics` computes the underlying signals that explain *why*, and -- more importantly for valuation -- which of it tends to repeat season over season. These metrics are wide-receiver-focused (the plan's original scope), split into three groups: opportunity **volume**, route-participation **efficiency**, and **touchdown** regression. Mirrors the [Metrics](https://nolmacdonald.github.io/nuclearff/tutorial/06_metrics.html) docs page.

## Volume: target share, air yards, and WOPR

`target_share`, `air_yards_share`, and `wopr` all require **weekly-grain** data -- "team" is only unambiguous week to week, since a traded player has a different team in different weeks. Passing seasonal data raises `ValueError` rather than silently dividing by the wrong team total. `racr` and `adot` are simple per-row ratios and accept either grain.

In [1]:
import polars as pl

from nuclearff.metrics import adot, air_yards_share, racr, target_share, wopr
from nuclearff.nflverse import load_weekly_receiving

weekly = load_weekly_receiving([2024])
df = target_share(weekly)
df = air_yards_share(df)
df = wopr(df)
df = racr(df)
df = adot(df)

df.group_by(["player_display_name", "position"]).agg(
    [
        pl.col("target_share").mean(),
        pl.col("air_yards_share").mean(),
        pl.col("wopr").mean(),
        pl.col("racr").mean(),
        pl.col("adot").mean(),
    ]
).sort("wopr", descending=True).head(6)

air_yards_share: recomputed values disagree with nflreadpy's own 'air_yards_share' column by more than 0.02 on 398/5444 rows (7.3%). A real disagreement this large usually means something subtle changed in nflreadpy's own methodology - worth investigating rather than assuming this recompute is wrong.


player_display_name,position,target_share,air_yards_share,wopr,racr,adot
str,str,f64,f64,f64,f64,f64
"""Malik Nabers""","""WR""",0.357041,0.465151,0.861167,0.86445,9.400565
"""A.J. Brown""","""WR""",0.321681,0.516489,0.844063,0.856395,12.916325
"""Courtland Sutton""","""WR""",0.271612,0.489094,0.749784,0.636096,13.408807
"""George Pickens""","""WR""",0.268305,0.468648,0.730511,0.706269,13.638393
"""Rashid Shaheed""","""WR""",0.247076,0.501651,0.72177,0.607442,18.218723
"""Justin Jefferson""","""WR""",0.298409,0.379181,0.71304,0.952639,11.384489


`WOPR` (Weighted Opportunity Rating, Hermsmeyer's formula: `1.5 * target_share + 0.7 * air_yards_share`) is the single best-fitting volume composite -- a receiver can lead 2024 WOPR with a lower target share than another player, purely because their share of their team's *air yards* is higher.

**Every volume function above recomputes its metric from raw columns rather than trusting nflreadpy's own precomputed ones**, then cross-checks the two and logs a warning on real disagreement. Confirmed on live 2024 data: the recomputed `target_share` matches nflreadpy's own column almost exactly, but `air_yards_share` disagrees on roughly 7% of rows -- nflreadpy's own denominator appears to draw from a richer play-by-play source than this project's per-player stats table captures. That's the cross-check doing its job, not a bug: trust the warning as a real data-quality signal, not noise to silence.

> **Why this matters:** a 7% disagreement sounds alarming until you remember what the cross-check is actually for: it exists so *you* notice a real, small, explainable gap between two independently computed numbers, rather than silently trusting whichever one happened to be computed first. If you ever see this warning fire on a metric you didn't expect, that's the signal to go check the two source columns by hand before trusting either one downstream -- not to suppress the warning.

## Efficiency: yards and targets per route run

Target share doesn't say how *hard* a receiver had to work for those targets. `yprr` (yards per route run) and `tprr` (targets per route run) need routes-run data, which isn't in the same table as receiving stats -- join them first with `join_routes`, which bridges the two via each player's `pfr_id`/`gsis_id`:

In [2]:
from nuclearff.metrics import join_routes, tprr, yprr
from nuclearff.nflverse import load_players, load_routes, load_seasonal_receiving

receiving = load_seasonal_receiving([2024])
routes = load_routes([2024])
players = load_players()

joined = join_routes(receiving, routes, players)
df = tprr(yprr(joined))

Both are **sample-gated**: below `min_routes` (200 by default), the result is `null` rather than a noisy small-sample number:

In [3]:
qualified = df.filter(pl.col("yprr").is_not_null())
print(f"{qualified.height} of {df.height} players ran >= 200 routes")

qualified.sort("yprr", descending=True).select(
    ["player_display_name", "position", "routes_run", "yprr", "tprr"]
).head(6)

293 of 511 players ran >= 200 routes


player_display_name,position,routes_run,yprr,tprr
str,str,f64,f64,f64
"""Puka Nacua""","""WR""",595.0,1.663866,0.178151
"""Ja'Marr Chase""","""WR""",1054.0,1.620493,0.166034
"""Nico Collins""","""WR""",622.0,1.617363,0.159164
"""Brian Thomas Jr.""","""WR""",821.0,1.56151,0.161998
"""Marvin Mims Jr.""","""WR""",326.0,1.542945,0.159509
"""Ladd McConkey""","""WR""",785.0,1.463694,0.142675


`routes_run` is honestly labeled by source (`routes_source` on the joined frame) -- nflverse doesn't publish a direct routes-run column for every season, so this is an approximation in some cases, not a guarantee of exactness. `ambiguous_player_id_pairs` mirrors `04_capturing_a_league.ipynb`'s Sleeper crosswalk check for this join's own `pfr_id`/`gsis_id` bridge.

## Touchdowns: expected TDs and regression

Touchdowns are the least stable part of a receiver's fantasy output year to year -- a receiver can lead the league in expected scoring opportunities and still finish with a below-average touchdown total, purely from variance. `expected_tds` joins ffverse's own expected-fantasy-opportunity model onto real receiving stats and adds `td_regression` (real minus expected):

In [4]:
from nuclearff.metrics import expected_tds
from nuclearff.nflverse import load_ff_opportunity, load_seasonal_receiving

receiving = load_seasonal_receiving([2024])
opportunity = load_ff_opportunity([2024])
df = expected_tds(receiving, opportunity)

df.filter(pl.col("expected_tds").is_not_null()).sort("td_regression").select(
    [
        "player_display_name",
        "position",
        "receiving_tds",
        "expected_tds",
        "td_regression",
    ]
).head(5)

expected_tds: 30 of 511 row(s) in `df` had no matching opportunity data - expected_tds/td_regression left null for those rows rather than dropped or erroring.


player_display_name,position,receiving_tds,expected_tds,td_regression
str,str,i32,f64,f64
"""Trey McBride""","""TE""",2,8.55,-6.55
"""Rome Odunze""","""WR""",3,7.07,-4.07
"""Adonai Mitchell""","""WR""",0,3.56,-3.56
"""George Pickens""","""WR""",3,6.46,-3.46
"""Travis Kelce""","""TE""",3,6.28,-3.28


Negative `td_regression` is the "TD-unlucky" list -- a player who scored fewer touchdowns than their real opportunities predicted, which is exactly the kind of gap a projection should expect to *close* next season, not repeat. Sort descending instead for the "TD-lucky" list -- players whose touchdown total is more likely to regress down. A player with no matching opportunity data gets `expected_tds = null` rather than being dropped or raising.

## What's Next

Volume, efficiency, and touchdown regression are all inputs, not a finished number. `07_projections.ipynb` blends real historical seasons -- weighted toward the two metrics above that repeat, and away from touchdown luck -- into a single forward-looking estimate.

## See Also

- `05_scoring_engine.ipynb` -- turns these same stats into fantasy points instead of opportunity signals.
- [Metrics](https://nolmacdonald.github.io/nuclearff/tutorial/06_metrics.html) -- the matching docs page.